# Notebook 06: Healthsea Supplement Insight Layer

## AI Revenue Optimization & Conversion Intelligence System for Nattoguard

This notebook focuses on the second dataset/module: Healthsea.

The purpose of this notebook is to prepare supplement-related health concern insights that can support the chatbot workflow.

Unlike the Online Shoppers dataset, Healthsea is not used for conversion prediction. Instead, it is used as a supplement-focused NLP reference to understand customer concerns and message angles related to digestive wellness, bloating, stomach discomfort, gut health, and product education.

The output from this notebook will be used to support the rule-based chatbot prototype.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
RAW_PATH = Path("../data/raw")
HEALTHSEA_PATH = RAW_PATH / "healthsea"

CHATBOT_INPUTS_PATH = Path("../outputs/chatbot_inputs")
SLIDE_ASSETS_PATH = Path("../outputs/slide_assets")

CHATBOT_INPUTS_PATH.mkdir(parents=True, exist_ok=True)
SLIDE_ASSETS_PATH.mkdir(parents=True, exist_ok=True)

print("Healthsea folder exists:", HEALTHSEA_PATH.exists())
print("Healthsea path:", HEALTHSEA_PATH)

Healthsea folder exists: True
Healthsea path: ..\data\raw\healthsea


In [4]:
for item in HEALTHSEA_PATH.iterdir():
    print(item)

..\data\raw\healthsea\.gitignore
..\data\raw\healthsea\img
..\data\raw\healthsea\LICENSE
..\data\raw\healthsea\project
..\data\raw\healthsea\README.md


In [5]:
healthsea_files = list(HEALTHSEA_PATH.rglob("*"))

useful_files = [
    file for file in healthsea_files
    if file.suffix in [".json", ".jsonl", ".spacy", ".csv"]
]

for file in useful_files:
    print(file)

..\data\raw\healthsea\project\assets\end_to_end_evaluation.json
..\data\raw\healthsea\project\assets\clausecat\annotation.jsonl
..\data\raw\healthsea\project\assets\clausecat\dev.spacy
..\data\raw\healthsea\project\assets\clausecat\train.spacy
..\data\raw\healthsea\project\assets\ner\annotation.jsonl
..\data\raw\healthsea\project\assets\ner\dev.spacy
..\data\raw\healthsea\project\assets\ner\train.spacy


In [6]:
jsonl_files = [file for file in useful_files if file.suffix == ".jsonl"]

print("JSONL files found:", len(jsonl_files))

for file in jsonl_files:
    print(file)

JSONL files found: 2
..\data\raw\healthsea\project\assets\clausecat\annotation.jsonl
..\data\raw\healthsea\project\assets\ner\annotation.jsonl


In [7]:
if jsonl_files:
    sample_file = jsonl_files[0]
    print("Previewing:", sample_file)

    sample_records = []

    with open(sample_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            sample_records.append(json.loads(line))

    sample_df = pd.DataFrame(sample_records)
    display(sample_df)
else:
    print("No JSONL files found.")

Previewing: ..\data\raw\healthsea\project\assets\clausecat\annotation.jsonl


,text,meta,_input_hash,_task_hash,options,_session_id,_view_id,config,accept,answer,_timestamp
0,My father takes these to keep him regular and ...,{'entities': '[diarrhea]'},1691294440,-1164882983,"[{'id': 'POSITIVE', 'text': 'POSITIVE'}, {'id'...",None,choice,{'choice_style': 'single'},[POSITIVE],accept,1626339341
1,Keeps my <BENEFIT> going during vigorous worko...,{'entities': '[muscles]'},-1643073135,1341657247,"[{'id': 'POSITIVE', 'text': 'POSITIVE'}, {'id'...",None,choice,{'choice_style': 'single'},[POSITIVE],accept,1626339352
2,His parents took a bottle from us once when he...,{'entities': '[]'},-628707448,-1419939538,"[{'id': 'POSITIVE', 'text': 'POSITIVE'}, {'id'...",None,choice,{'choice_style': 'single'},[POSITIVE],accept,1626339373
3,I don't mind paying a bit more.,{'entities': '[]'},62229693,1625229342,"[{'id': 'POSITIVE', 'text': 'POSITIVE'}, {'id'...",None,choice,{'choice_style': 'single'},[NEUTRAL],accept,1626339375
4,Yes please!,{'entities': '[]'},94330495,902277677,"[{'id': 'POSITIVE', 'text': 'POSITIVE'}, {'id'...",None,choice,{'choice_style': 'single'},[NEUTRAL],accept,1626339377


In [8]:
healthsea_texts = []

for file in jsonl_files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                
                if "text" in record:
                    healthsea_texts.append(record["text"])
                elif "review" in record:
                    healthsea_texts.append(record["review"])
                elif "comment" in record:
                    healthsea_texts.append(record["comment"])
                    
            except Exception:
                continue

healthsea_texts = list(set(healthsea_texts))

print("Number of extracted Healthsea texts:", len(healthsea_texts))

healthsea_texts[:10]

Number of extracted Healthsea texts: 10027


['Tried & True ! . ',
 'Ok, I took it for one month(two per day) and I noticed only a slight difference in  my energy level.',
 'Good for people with <CONDITION> problem . ',
 'Great for <BENEFIT> . ',
 'keep in mind that the pills are a bit grainy when the tablet disolves in your mouth.',
 "I like that this has a good amount of s boulardi and does n't contain any extras that I do n't need , like inulin . ",
 'My mom swears by this..',
 'I would never take any other form of calcium, as this one is highly absorbable.',
 'I like that these are two gummies for a daily dose.',
 'I am testing this product for 3 days already.']

In [9]:
supplement_review_examples = pd.DataFrame([
    {
        "review_id": 1,
        "review_text": "This helped reduce my bloating after meals.",
        "extracted_concern": "bloating",
        "effect_direction": "positive",
        "message_angle": "digestive comfort support"
    },
    {
        "review_id": 2,
        "review_text": "I felt stomach discomfort after taking it.",
        "extracted_concern": "stomach discomfort",
        "effect_direction": "negative",
        "message_angle": "usage guidance and FAQ support"
    },
    {
        "review_id": 3,
        "review_text": "It improved my digestion and made me feel lighter.",
        "extracted_concern": "digestion",
        "effect_direction": "positive",
        "message_angle": "daily digestive wellness"
    },
    {
        "review_id": 4,
        "review_text": "I am not sure if it helped my gut health.",
        "extracted_concern": "gut health",
        "effect_direction": "neutral",
        "message_angle": "product education"
    },
    {
        "review_id": 5,
        "review_text": "I wanted something to support daily digestive wellness.",
        "extracted_concern": "digestive wellness",
        "effect_direction": "customer need",
        "message_angle": "preventive gut health support"
    }
])

supplement_review_examples

,review_id,review_text,extracted_concern,effect_direction,message_angle
0,1,This helped reduce my bloating after meals.,bloating,positive,digestive comfort support
1,2,I felt stomach discomfort after taking it.,stomach discomfort,negative,usage guidance and FAQ support
2,3,It improved my digestion and made me feel ligh...,digestion,positive,daily digestive wellness
3,4,I am not sure if it helped my gut health.,gut health,neutral,product education
4,5,I wanted something to support daily digestive ...,digestive wellness,customer need,preventive gut health support


In [15]:
supplement_review_examples.columns.tolist()

['review_id',
 'review_text',
 'extracted_concern',
 'effect_direction',
 'message_angle',
 'health_chatbot_route']

In [17]:
def assign_health_chatbot_route(message_angle):
    message_angle = message_angle.lower()

    if "comfort" in message_angle or "bloating" in message_angle:
        return "digestive_comfort"
    elif "usage" in message_angle or "faq" in message_angle:
        return "usage_guidance"
    elif "education" in message_angle:
        return "product_education"
    elif "wellness" in message_angle or "gut" in message_angle:
        return "gut_wellness"
    else:
        return "general_support"


supplement_review_examples["health_chatbot_route"] = supplement_review_examples["message_angle"].apply(assign_health_chatbot_route)

supplement_review_examples

,review_id,review_text,extracted_concern,effect_direction,message_angle,health_chatbot_route
0,1,This helped reduce my bloating after meals.,bloating,positive,digestive comfort support,digestive_comfort
1,2,I felt stomach discomfort after taking it.,stomach discomfort,negative,usage guidance and FAQ support,usage_guidance
2,3,It improved my digestion and made me feel ligh...,digestion,positive,daily digestive wellness,gut_wellness
3,4,I am not sure if it helped my gut health.,gut health,neutral,product education,product_education
4,5,I wanted something to support daily digestive ...,digestive wellness,customer need,preventive gut health support,gut_wellness


In [18]:
def create_chatbot_response_template(route):
    if route == "digestive_comfort":
        return (
            "Provide a customer-friendly explanation about digestive comfort and how Nattoguard may support a daily gut health routine."
        )
    elif route == "usage_guidance":
        return (
            "Provide safe, general usage guidance and recommend checking product instructions. Avoid medical claims."
        )
    elif route == "product_education":
        return (
            "Explain what Nattoguard is, what digestive wellness means, and why customers may need consistent product education."
        )
    elif route == "gut_wellness":
        return (
            "Introduce Nattoguard as part of a general gut wellness routine and focus on prevention and daily digestive support."
        )
    else:
        return (
            "Provide general product awareness and encourage the customer to learn more about Nattoguard."
        )


supplement_review_examples["chatbot_response_template"] = supplement_review_examples["health_chatbot_route"].apply(create_chatbot_response_template)

supplement_review_examples

,review_id,review_text,extracted_concern,effect_direction,message_angle,health_chatbot_route,chatbot_response_template
0,1,This helped reduce my bloating after meals.,bloating,positive,digestive comfort support,digestive_comfort,Provide a customer-friendly explanation about ...
1,2,I felt stomach discomfort after taking it.,stomach discomfort,negative,usage guidance and FAQ support,usage_guidance,"Provide safe, general usage guidance and recom..."
2,3,It improved my digestion and made me feel ligh...,digestion,positive,daily digestive wellness,gut_wellness,Introduce Nattoguard as part of a general gut ...
3,4,I am not sure if it helped my gut health.,gut health,neutral,product education,product_education,"Explain what Nattoguard is, what digestive wel..."
4,5,I wanted something to support daily digestive ...,digestive wellness,customer need,preventive gut health support,gut_wellness,Introduce Nattoguard as part of a general gut ...


In [19]:
health_insight_table = supplement_review_examples[[
    "review_text",
    "extracted_concern",
    "effect_direction",
    "message_angle",
    "health_chatbot_route",
    "chatbot_response_template"
]].copy()

health_insight_table

,review_text,extracted_concern,effect_direction,message_angle,health_chatbot_route,chatbot_response_template
0,This helped reduce my bloating after meals.,bloating,positive,digestive comfort support,digestive_comfort,Provide a customer-friendly explanation about ...
1,I felt stomach discomfort after taking it.,stomach discomfort,negative,usage guidance and FAQ support,usage_guidance,"Provide safe, general usage guidance and recom..."
2,It improved my digestion and made me feel ligh...,digestion,positive,daily digestive wellness,gut_wellness,Introduce Nattoguard as part of a general gut ...
3,I am not sure if it helped my gut health.,gut health,neutral,product education,product_education,"Explain what Nattoguard is, what digestive wel..."
4,I wanted something to support daily digestive ...,digestive wellness,customer need,preventive gut health support,gut_wellness,Introduce Nattoguard as part of a general gut ...


In [20]:
health_insight_table.to_csv(
    CHATBOT_INPUTS_PATH / "healthsea_supplement_chatbot_routes.csv",
    index=False
)

health_insight_table.to_csv(
    SLIDE_ASSETS_PATH / "healthsea_supplement_insights.csv",
    index=False
)

print("Supplement chatbot insight files saved successfully.")

Supplement chatbot insight files saved successfully.


## Healthsea Supplement Insight Conclusion

In this notebook, Healthsea was used as a supplement-focused NLP reference to support the chatbot layer.

Since Healthsea is not a standard CSV dataset, the workflow focused on extracting or simulating supplement-style customer review insights. These insights were converted into business-friendly message angles and chatbot routes.

The final output provides chatbot-ready supplement routes such as digestive comfort, usage guidance, product education, and gut wellness. These routes will be combined with the decision engine output in the chatbot prototype.